In [36]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np

dataset=fetch_california_housing()

In [37]:
dataset

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
           37.88      , -122.23      ],
        [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
           37.86      , -122.22      ],
        [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
           37.85      , -122.24      ],
        ...,
        [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
           39.43      , -121.22      ],
        [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
           39.43      , -121.32      ],
        [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
           39.37      , -121.24      ]]),
 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894]),
 'frame': None,
 'target_names': ['MedHouseVal'],
 'feature_names': ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude'],
 'DESCR': '.. _california_housing_dataset:\n

In [38]:
df=pd.DataFrame(data=dataset.data,columns=dataset.feature_names)

In [39]:
df["MedHouseVal"]=dataset.target

In [27]:
df.head()
df.shape

(20640, 9)

In [47]:
X=df.drop("MedHouseVal",axis=1)
y=df["MedHouseVal"]

In [48]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.30,random_state=3)

In [49]:
X_train.shape,X_test.shape

((14448, 8), (6192, 8))

In [42]:
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

In [51]:
def perform(tru,pred):
    mae=mean_absolute_error(tru,pred)
    mse=mean_squared_error(tru,pred)
    rmse=np.sqrt(mse)
    score=r2_score(tru,pred)
    print("mean absolute error is",mae)
    print("mean squared error is",mse)
    print("root mean squared error is",rmse)
    print(" r2 score",score)
    print('-'*50)

In [52]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import AdaBoostRegressor
from xgboost import XGBRegressor

models={
    "GradientBoostingRegressor":GradientBoostingRegressor(),
    "AdaBoostRegressor":AdaBoostRegressor(),
    "XGBRegressor":XGBRegressor()
}

for name,model in models.items():
    model.fit(X_train,y_train)
    y_pred=model.predict(X_test)
    print(f"using {name}")
    perform(y_test,y_pred)

using GradientBoostingRegressor
mean absolute error is 0.3635018827196565
mean squared error is 0.27605197726771663
root mean squared error is 0.5254064876528616
 r2 score 0.7879150632688782
--------------------------------------------------
using AdaBoostRegressor
mean absolute error is 0.7340623142922964
mean squared error is 0.7285960765742872
root mean squared error is 0.8535783950957798
 r2 score 0.44023493570943395
--------------------------------------------------
using XGBRegressor
mean absolute error is 0.3092206204963739
mean squared error is 0.2197111082350276
root mean squared error is 0.4687335151608295
 r2 score 0.8312005697247368
--------------------------------------------------


## Hyperparameter Tuning

In [ ]:
gb={
    "loss" : ['log_loss', 'exponential'],
    "n_estimators":[100,200,500],
    "criterion" : ['friedman_mse', 'squared_error']
}
ab={
    "algorithm" : ['SAMME', 'SAMME.R'],
    "n_estimators":[100,200,500]
}
xgb={
    "n_estimators": [100, 200, 500],
    "learning_rate": [0.01, 0.1, 0.2, 0.3],
    "max_depth": [3, 5, 7],
    "subsample": [0.8, 1.0]
}

hyper_models=[
    ["GradientBoostingClassifier",GradientBoostingClassifier(),gb],
    ["AdaBoostClassifier",AdaBoostClassifier(),ab],
    ["XGBClassifier",XGBClassifier(),xgb]
]

from sklearn.model_selection import RandomizedSearchCV

for name,model,param in hyper_models:
    randomcv=RandomizedSearchCV(estimator=model,param_distributions=param,cv=3,n_jobs=-1,verbose=1)
    randomcv.fit(X_train,y_train)

    print(randomcv.best_params_)
    print("using",name)
    print("for train")
    perform(y_train, randomcv.predict(X_train))
    print("for test")
    y_pred=randomcv.predict(X_test)
    perform(y_test,y_pred)